## Загрузка датасета MMReD на HuggingFace

Этот ноутбук загружает датасет MMReD на HuggingFace Hub в репозиторий `dondosss/mmred_mera`.
Запускайте ячейки последовательно.

In [ ]:
import json
import os
import datasets
from tqdm import tqdm

### Настройка путей

MMReD состоит из 15 поднаборов (5 типов задач × 3 длины последовательности).
Каждый поднабор хранится в своём JSON-файле.

`path_to_data` — папка, содержащая папки поднаборов (например, `mmred_dc_sa_c_32/test.json`).
`path_to_meta` — путь до `raw_dataset_meta.json`.

In [ ]:
# Путь до сгенерированных данных (папка, содержащая поднаборы или плоские файлы)
path_to_data = "./"
path_to_meta = "./"

### Флаг приватности

MMReD — публичный датасет (ответы в test не скрываются).
Если нужно скрыть ответы, установите `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True`.

In [ ]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = False

### Вспомогательные функции

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def hide_answers(dataset_split):
    for card in tqdm(dataset_split):
        card["outputs"] = ""

### Список поднаборов

In [ ]:
TASK_TYPES = ["dc_sa_c", "dc_sr_i", "dc_cc_i", "dc_ws_r", "dc_whs_c"]
SEQ_LENS = [32, 64, 128]

SUBSETS = [f"mmred_{task}_{seq_len}" for task in TASK_TYPES for seq_len in SEQ_LENS]
print(f"Total subsets: {len(SUBSETS)}")
print(SUBSETS)

### Загрузка метаданных и промптов

In [ ]:
meta = load_json(os.path.join(path_to_meta, "raw_dataset_meta.json"))
prompts = meta["prompts"]
print(f"Loaded {len(prompts)} prompts")

### Определение features датасета

In [ ]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "context": datasets.Value("string"),
        "question": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "categories": {
            "task_type": datasets.Value("string"),
            "seq_len": datasets.Value("int32"),
            "atype": datasets.Value("string"),
        },
    },
})

### Загрузка и обработка каждого поднабора

Каждый поднабор хранится в папке `<subset_name>/` внутри `path_to_data`.
Ожидаемая структура:
```
path_to_data/
  mmred_dc_sa_c_32/
    test.json      # {"data": [...]}
    shots.json     # {"data": [...]}
  mmred_dc_sa_c_64/
    ...
```

Если данные лежат плоско в одном JSON — адаптируйте путь ниже.

In [ ]:
all_dataset_dicts = {}

for subset in tqdm(SUBSETS, desc="Processing subsets"):
    subset_dir = os.path.join(path_to_data, subset)
    test_path = os.path.join(subset_dir, "test.json")
    shots_path = os.path.join(subset_dir, "shots.json")

    if not os.path.exists(test_path):
        print(f"WARNING: {test_path} not found — skipping {subset}")
        continue

    test = load_json(test_path)["data"]
    shots = load_json(shots_path)["data"] if os.path.exists(shots_path) else []

    # Replace instruction index with actual prompt string
    for card in shots:
        card["instruction"] = prompts[card["instruction"]]
    for card in test:
        card["instruction"] = prompts[card["instruction"]]

    # Optionally hide answers for private datasets
    if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
        hide_answers(test)

    # Build HuggingFace dataset splits
    shots_ds = datasets.Dataset.from_list(shots, features=features) if shots else None

    STEP = 50
    chunks = []
    for i in range(0, len(test), STEP):
        chunk = datasets.Dataset.from_list(test[i:i + STEP], features=features)
        chunks.append(chunk)
    test_ds = datasets.concatenate_datasets(chunks) if chunks else datasets.Dataset.from_list([], features=features)

    splits = {"test": test_ds}
    if shots_ds is not None:
        splits["shots"] = shots_ds

    all_dataset_dicts[subset] = datasets.DatasetDict(splits)
    print(f"{subset}: test={len(test_ds)}, shots={len(shots_ds) if shots_ds else 0}")

print(f"\nReady to upload {len(all_dataset_dicts)} subsets.")

### Проверка данных перед загрузкой

In [ ]:
# Выберите любой поднабор для проверки
sample_subset = "mmred_dc_sa_c_32"
if sample_subset in all_dataset_dicts:
    ds = all_dataset_dicts[sample_subset]
    print(ds)
    print("\nFirst test sample:")
    print(json.dumps(ds["test"][0], ensure_ascii=False, indent=2))

### Загрузка на HuggingFace Hub

In [ ]:
### TOKEN — замените на свой HF write-token
token = "hf_YOUR_TOKEN_HERE"

### Репозиторий назначения
REPO_ID = "dondosss/mmred_mera"

In [ ]:
for subset, dataset_dict in tqdm(all_dataset_dicts.items(), desc="Uploading subsets"):
    print(f"Uploading {subset} ...")
    dataset_dict.push_to_hub(
        REPO_ID,
        config_name=subset,
        private=True,
        token=token,
    )
    print(f"  Done: {REPO_ID} / {subset}")

print("\nAll subsets uploaded.")

### Проверка загруженных данных

In [ ]:
# Загружаем один поднабор с HF для проверки
check_subset = "mmred_dc_sa_c_32"
ds_hf = datasets.load_dataset(REPO_ID, name=check_subset, token=token)
print(ds_hf)

In [ ]:
# Проверяем, что все id совпадают с оригиналом
orig_test = load_json(os.path.join(path_to_data, check_subset, "test.json"))["data"]

bools = [orig_test[i]["meta"]["id"] == ds_hf["test"][i]["meta"]["id"] for i in range(len(orig_test))]
print(f"IDs match: {all(bools)}")
print(f"Counts match: {len(orig_test) == len(ds_hf['test'])}")

In [ ]:
# Проверяем вопросы
for card in ds_hf["test"]:
    assert isinstance(card["inputs"]["context"], str), "context should be string"
    assert isinstance(card["inputs"]["question"], str), "question should be string"
print("All fields validated.")